# Topology-Aware Quantum Kernel

This notebook exercises the public `topology_kernel_product` facade with exact local statevectors. It performs no provider or QPU call. The frozen labels come from the same ring kernel evaluated below, so the result demonstrates representability of that declared inductive bias, not independent generalisation or quantum advantage.

In [ ]:
import numpy as np

from scpn_quantum_control.topology_kernel_product import (
    TOPOLOGY_KERNEL_CLAIM_BOUNDARY,
    TopologyKernelConfig,
    build_teacher_aligned_dataset,
    build_topology_kernel_evidence,
    evaluate_kernel_ridge,
    fidelity_kernel_matrix,
    fit_kernel_ridge,
    ring_topology,
)

print(TOPOLOGY_KERNEL_CLAIM_BOUNDARY)

## Frozen teacher-aligned split

For four nodes there are six canonical undirected edges. Seed 880 draws two prototypes and a candidate pool, ranks candidates by their ring-kernel prototype-similarity difference, and returns balanced, disjoint 32/16 train/test splits.

In [ ]:
config = TopologyKernelConfig()
dataset = build_teacher_aligned_dataset(config, seed=880)
assert set(dataset.train_ids).isdisjoint(dataset.test_ids)
assert dataset.train_labels.tolist() == [1, -1] * 16
print(
    {
        "train_shape": dataset.train_features.shape,
        "test_shape": dataset.test_features.shape,
        "dataset_digest": dataset.content_digest,
    }
)

## Exact ring fidelity Gram matrix

Each feature modulates only its matching upper-triangle XY coupling. The Gram matrix must be symmetric, have unit diagonal to numerical tolerance, and be positive semidefinite to numerical tolerance.

In [ ]:
topology = ring_topology(config.n_qubits)
train_kernel = fidelity_kernel_matrix(
    dataset.train_features,
    dataset.train_features,
    topology,
    config,
    row_ids=dataset.train_ids,
    column_ids=dataset.train_ids,
)
eigenvalues = np.linalg.eigvalsh((train_kernel.values + train_kernel.values.T) / 2.0)
assert np.allclose(train_kernel.values, train_kernel.values.T, atol=1e-12, rtol=0.0)
assert np.allclose(np.diag(train_kernel.values), 1.0, atol=1e-12, rtol=0.0)
print({"minimum_eigenvalue": float(eigenvalues.min()), "gram_digest": train_kernel.content_digest})

## Identifier-bound kernel ridge classification

The cross-kernel columns must exactly match the fitted training identifiers and topology digest. This prevents silent coefficient reordering or cross-topology prediction.

In [ ]:
test_kernel = fidelity_kernel_matrix(
    dataset.test_features,
    dataset.train_features,
    topology,
    config,
    row_ids=dataset.test_ids,
    column_ids=dataset.train_ids,
)
model = fit_kernel_ridge(train_kernel, dataset.train_labels, alpha=config.ridge)
result = evaluate_kernel_ridge("ring", model, test_kernel, dataset.test_labels)
assert result.correct == 16
print({"correct": result.correct, "total": result.total, "accuracy": result.accuracy})

## Frozen controls and interpretation

The evidence builder fits separate ring, path, complete, zero-coupling, and classical RBF kernels on the same split. Separation from these controls is specific to this circular teacher-aligned task and does not establish quantum advantage or domain value.

In [ ]:
evidence = build_topology_kernel_evidence(config=config, seed=880)
summary = {
    name: getattr(evidence, name).accuracy
    for name in ("ring", "path", "complete", "zero", "classical_rbf")
}
assert summary == {
    "ring": 1.0,
    "path": 0.25,
    "complete": 0.5625,
    "zero": 0.5,
    "classical_rbf": 0.5,
}
print(summary)
print(
    {
        "evidence_digest": evidence.content_digest,
        "relabel_error": evidence.permutation_max_abs_error,
    }
)

Application-domain transfer remains explicitly descoped because no typed domain-kit consumer is implemented. See `docs/topology_aware_quantum_kernel.md` for the full API, controls, scientific basis, and non-claims.